Lab 2: ImageFolder & DataLoaders

Now that we've explored our image data, it's time to transform it into a format PyTorch can work with. In this lab, we'll learn how to use

torchvision.transforms

to preprocess images and

ImageFolder

with

DataLoader

to efficiently load batches of data.

Project Overview: PyTorch Custom Datasets

![image](https://raw.githubusercontent.com/poridhiEng/lab-asset/c05b0fcffc73075ac57eef168b7647e050b9156b/tensorcode/Deep-learning-with-pytorch/Custom-Datasets-with-pytorch/Lab_01/images/lab1.svg)

This project is split into

4 labs

:

Lab 1: Data Preparation & Exploration

— Download and visualize custom data

(Current Lab) Lab 2: ImageFolder & DataLoaders

— Transform and load images efficiently

Lab 3: Custom Datasets

— Build your own Dataset class from scratch

Lab 4: TinyVGG Model Training

— Create and train a CNN model

**Goal of This Lab**

In this lab, we focus on transforming data and loading it efficiently for model training.

What we'll learn:

* How to use `torchvision.transforms` to preprocess images
* How to compose multiple transforms together
* How to load images with ImageFolder
* How to create iterable batches with DataLoader

## Part 1: Why Transform Data?

### The Problem

Our raw images have several issues when training a neural network. These problems must be fixed before feeding the images into a model.

| Issue  | Problem | Solution |
|------|------|------|
| Format | Images are stored as PIL objects or NumPy arrays | Convert them to PyTorch tensors |
| Size | Images may have different dimensions | Resize all images to a uniform size |
| Scale | Pixel values range from 0–255 | Normalize the values to 0–1 |
| Type | Pixel values are integers | Convert them to float32 |




---



এখানে মূল কথা হলো **নিউরাল নেটওয়ার্কে ছবি দেওয়ার আগে ছবিকে একটু প্রসেস (transform) করতে হয়**। কারণ raw image সরাসরি দিলে মডেল ঠিকভাবে কাজ করতে পারে না। নিচে সহজভাবে বুঝাই।

### 1️⃣ Format (ফরম্যাট সমস্যা)

**Problem:**
অনেক সময় ছবি **PIL image** বা **NumPy array** আকারে থাকে। কিন্তু **PyTorch** সরাসরি এগুলো দিয়ে কাজ করতে পারে না।

**Solution:**
ছবিকে **Tensor** এ convert করতে হয়।

👉 সহজভাবে:
Image → Tensor

---

### 2️⃣ Size (সাইজ সমস্যা)

**Problem:**
সব ছবির সাইজ একরকম না।
একটা ছবি হতে পারে **200×300**, আরেকটা **500×500**।

কিন্তু নিউরাল নেটওয়ার্কে সব ইনপুটের সাইজ **একই হতে হয়**।

**Solution:**
সব ছবি **একই সাইজে resize** করা হয়।
যেমন: 224×224

---

### 3️⃣ Scale (পিক্সেল ভ্যালু সমস্যা)

**Problem:**
ছবির pixel value সাধারণত **0 থেকে 255** এর মধ্যে থাকে।

কিন্তু মেশিন লার্নিং মডেল ছোট মানে ভালোভাবে শিখে।

**Solution:**
পিক্সেলকে **0–1 এর মধ্যে normalize** করা হয়।

👉 উদাহরণ
255 → 1
128 → 0.5

মানে
pixel / 255

---

### 4️⃣ Type (ডেটা টাইপ সমস্যা)

**Problem:**
ছবির pixel value সাধারণত **integer** হয় (যেমন 120, 200)।

কিন্তু neural network সাধারণত **float32** টাইপে কাজ করে।

**Solution:**
pixel values কে **float32** এ convert করা হয়।




The Solution: Transforms

![image](https://raw.githubusercontent.com/poridhiEng/lab-asset/c05b0fcffc73075ac57eef168b7647e050b9156b/tensorcode/Deep-learning-with-pytorch/Custom-Datasets-with-pytorch/Lab_02/images/img3.svg)

## Image Transformations using torchvision.transforms

The library torchvision.transforms provides a collection of common image transformations that are useful when preparing images for deep learning models in PyTorch.

### Common Transformations

- **Resize()**  
  Changes the dimensions of an image so that all images have the same size. Neural networks require a fixed input size.

- **ToTensor()**  
  Converts a PIL image or NumPy array into a PyTorch tensor and scales pixel values from the range **[0, 255]** to **[0, 1]**.

- **Normalize()**  
  Normalizes the tensor image using a specified **mean** and **standard deviation (std)**. This helps stabilize and speed up model training.

- **RandomHorizontalFlip()**  
  Randomly flips the image horizontally. This is a form of **data augmentation** that increases dataset diversity and helps reduce overfitting.

**Part 2: Composing Transforms**

We can chain multiple transforms together using Compose():

In [ ]:
from torchvision import transforms

data_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),              # ছবির সাইজ 64 × 64 pixel করা হবে।
    transforms.RandomHorizontalFlip(p=0.5),        # ছবি ৫০% সম্ভাবনায় mirror (left ↔ right flip) হবে।  এটা data augmentation
    transforms.ToTensor()                          # ছবিকে Tensor এ convert করবে এবং pixel value 0–255 → 0–1 এ নিয়ে আসবে।
])

## Understanding Each Transform

### 1. Resize

**transforms.Resize(size=(64, 64))**

- Resizes images to **64 × 64 pixels**.
- Ensures that all images have the **same dimensions**, which is required for neural networks.

**Trade-offs to consider:**

- **Smaller sizes (64×64)**  
  - Faster training  
  - Lower memory usage  
  - Less image detail  

- **Larger sizes (256×256)**  
  - More image detail  
  - Slower training  
  - Higher memory usage  

- An image resized from **512×512 to 64×64** loses about **98.4% of its pixel information**.

---

### 2. RandomHorizontalFlip

**transforms.RandomHorizontalFlip(p=0.5)**

- Randomly flips images **horizontally (mirror effect)**.
- `p = 0.5` means each image has a **50% chance of being flipped**.

This is a type of **data augmentation** used to improve model generalization.

**Why augmentation matters:**

- Artificially increases the **dataset size**
- Helps prevent **overfitting** (when the model memorizes training data)
- Makes the model **robust to variations** (e.g., an object facing left or right)

⚠️ Data augmentation should be applied **only to training data**, **not to test data**.

---

### 3. ToTensor

**transforms.ToTensor()**

- Converts a **PIL Image** into a **PyTorch tensor**.
- Changes the image shape from **[H, W, C]** to **[C, H, W]** (PyTorch format).
- Scales pixel values from **[0, 255]** to **[0.0, 1.0]**.
- Changes the data type from **uint8** to **float32**.

**Why normalize pixel values to [0, 1]?**

- Neural networks work better with **smaller numbers**
- Helps prevent **numerical instability during training**
- Makes **gradient descent more efficient**

## Part 3: Loading with ImageFolder

### What is ImageFolder?

`torchvision.datasets.ImageFolder` is a convenient dataset class in PyTorch that helps load image datasets easily.

It automatically:
- Loads images from a **directory structure**
- Assigns **labels based on folder names**
- Applies **transforms** to each image

In short:

ImageFolder(root, transform) → Dataset containing **(image, label)** pairs.

---

### Directory Structure Requirements

ImageFolder expects the dataset to be organized in the following directory structure:

root/

├── class_a/

│   ├── image1.jpg

│   └── image2.jpg

├── class_b/

│   └── ...

└── class_c/

    └── ...

Each **folder name represents a class label**, and the images inside that folder belong to that class.

## সহজভাবে ব্যাখ্যা

এখানে **PyTorch এর torchvision লাইব্রেরির `ImageFolder` ব্যবহার করা হয়**।

### ImageFolder কী করে

`ImageFolder` একটি dataset loader যা **ফোল্ডার দেখে নিজে নিজে label তৈরি করে**।

তুমি শুধু dataset এর **root folder** দিলে এটি:

1️⃣ সব ছবি load করবে
2️⃣ folder নাম দেখে **label assign করবে**
3️⃣ চাইলে **transform apply করবে**

---

### Directory Structure কেন দরকার

`ImageFolder` ঠিকমতো কাজ করতে dataset এ **class অনুযায়ী আলাদা folder থাকতে হবে**।

উদাহরণ:

```
dataset/
   cat/
      cat1.jpg
      cat2.jpg
   dog/
      dog1.jpg
      dog2.jpg
```

এখানে:

* `cat` folder → label **0**
* `dog` folder → label **1**

মানে:

```
cat1.jpg → (image, 0)
dog1.jpg → (image, 1)
```

---

✅ **সংক্ষেপে**

`ImageFolder` =
Folder structure → Image load → Label assign → Transform apply → `(image, label)`



**Using ImageFolder**

In [ ]:
from torchvision import datasets

train_data = datasets.ImageFolder(
    root=train_dir,                 # Training images যেই folder এ আছে তা path দিতে হবে।
    transform=data_transform        # আমরা আগেই তৈরি করা transform pipeline apply করব।
)

test_data = datasets.ImageFolder(
    root=test_dir,
    transform=data_transform        # test dataset এ data augmentation (RandomHorizontalFlip) সাধারণত apply করা হয় না, শুধু Resize + ToTensor করা হয়।
)

**Accessing Dataset Attributes**

ImageFolder provides useful attributes:

In [ ]:
# Get class names as a list
class_names = train_data.classes
# Output: ['pizza', 'steak', 'sushi']

# Get class-to-index mapping
class_dict = train_data.class_to_idx
# Output: {'pizza': 0, 'steak': 1, 'sushi': 2}

# Get dataset length
len(train_data)  # 225

- `classes` attribute provides a list of class names
- `class_to_idx` attribute maps class names to indices
- `len()` function returns the number of samples in the dataset

**Accessing Individual Samples**

In [ ]:
# Get a single sample (image tensor, label)
img, label = train_data[0]

print(f"Image shape: {img.shape}")  # [3, 64, 64]
print(f"Image dtype: {img.dtype}")  # torch.float32
print(f"Label: {label}")  # 0 (for pizza)

## Understanding the Output of ImageFolder

When we load an image from `ImageFolder` and apply transforms, the output looks like this:

- **Shape:** `[3, 64, 64]`  
  - `[Channels, Height, Width]`  
  - `3` channels for **RGB** (Red, Green, Blue)  
  - `64` pixels tall, `64` pixels wide  
  - Total values per image: `3 × 64 × 64 = 12,288`

- **dtype:** `torch.float32`  
  - 32-bit floating point numbers  
  - Converted from the original uint8 (0–255)

- **Label:** `0`  
  - Integer index representing the class  
  - Mapping: `0 = pizza`, `1 = steak`, `2 = sushi`

**Visualizing Transformed Images**

When plotting with matplotlib, remember to permute dimensions:

In [ ]:
import matplotlib.pyplot as plt

# PyTorch format: [C, H, W] -> [3, 64, 64]
# Matplotlib format: [H, W, C] -> [64, 64, 3]
img_permuted = img.permute(1, 2, 0)
                                          # 1, 2, 0 মানে:
                                          # Old dimension 1 → new 0 (Height)
                                          # Old dimension 2 → new 1 (Width)
                                          # Old dimension 0 → new 2 (Channels)

plt.imshow(img_permuted)
plt.title(f"Class: {class_names[label]}")
plt.axis('off')
plt.show()

**Image quality observation:**

Notice that 64x64 images appear more pixelated than the original 512x512 images. This is expected! If you find it harder to recognize the image, a neural network will likely struggle too. Finding the right balance between image size and computational efficiency is key.

ঠিক বলেছো! সহজভাবে বোঝাই:

---

### Image quality observation

* যখন **512×512 → 64×64** resize করা হয়, image **pixelated বা blurrier** দেখায়।
* ছোট size = **কম detail**, তাই মানুষের চোখেও অনেক detail হারায়।
* Neural network এর জন্যও একই কথা প্রযোজ্য:

  * **কম detail** → model কিছু subtle features ধরতে পারবে না
  * **আরো challenging task** → model ভালোভাবে recognize করতে struggle করতে পারে

---

### Balance is key

* **Bigger image** → more detail, accurate recognition, **slower training**, more memory usage
* **Smaller image** → faster training, less memory, কিন্তু **detail কম**



**Part 4: Creating DataLoaders**

Why DataLoaders?
A Dataset alone is not enough for efficient training. We need a DataLoader to:

- Creates batches of data for efficient training
- Shuffles data to prevent learning order ,         model যেন order মুখস্থ না করে
- Enables parallel loading with multiple workers
- Makes datasets iterable for training loops

DataLoader Workflow: Step-by-Step

![image](https://raw.githubusercontent.com/poridhiEng/lab-asset/c05b0fcffc73075ac57eef168b7647e050b9156b/tensorcode/Deep-learning-with-pytorch/Custom-Datasets-with-pytorch/Lab_02/images/img5.svg)

Let's understand what happens when we use a DataLoader:

Detailed Process Breakdown

0. Dataset with Individual Samples

In [ ]:
ImageFolder Dataset: 225 training images
├── Sample 0: (tensor[3,64,64], label=0)  # Pizza
├── Sample 1: (tensor[3,64,64], label=1)  # Steak
├── Sample 2: (tensor[3,64,64], label=2)  # Sushi
├── Sample 3: (tensor[3,64,64], label=0)  # Pizza
└── ... (221 more samples)

1. Shuffling Process

Without Shuffle (shuffle=False):

In [ ]:
# Samples processed in order
Epoch 1: [0, 1, 2, 3, ..., 224]
Epoch 2: [0, 1, 2, 3, ..., 224]  # Same order!

With Shuffle (shuffle=True):

In [ ]:
# Samples randomized each epoch
Epoch 1: [142, 7, 89, 3, ..., 56]
Epoch 2: [201, 15, 73, 128, ..., 9]  # Different order!

**Why shuffle matters:**

- Prevents model from learning sample order
- Each epoch sees different sample combinations
- Improves generalization

2. Batching Process

Example with batch_size=32 and 225 samples:

In [ ]:
Total samples: 225
Batch size: 32

Batch 1: samples [0-31]    → 32 images
Batch 2: samples [32-63]   → 32 images
    ...         ...             ...
Batch 7: samples [192-224] → 33 images (last batch is larger!)

Total batches per epoch: 7

Tensor Stacking:

In [ ]:
# Before batching: Individual tensors
image_0: [3, 64, 64]
image_1: [3, 64, 64]
...
image_31: [3, 64, 64]

# After batching: Stacked into one tensor
batch_images: [32, 3, 64, 64]  # New dimension added! Batch size (32 images)
batch_labels: [32]

3. Parallel Loading with num_workers

num_workers=0 (Single Process):

In [ ]:
Timeline:
[GPU: Process Batch 1] → [CPU: Load Batch 2] → [GPU: Process Batch 2] → [CPU: Load Batch 3]
         ↑ GPU idle while loading next batch ↑

num_workers=2 (Multi-Process):

In [ ]:
Timeline:
[GPU: Process Batch 1] ←→ [Worker 1: Load Batch 2]
                       ←→ [Worker 2: Load Batch 3]
                                ↑ Next batch ready when GPU finishes!

**Performance comparison:**

`num_workers=0:` CPU loads data → GPU processes → CPU loads next → repeat
`num_workers=2+:` CPU loads next batch while GPU processes current batch
Result: Significant speedup (GPU stays busy, less idle time)

### After Processing All Batches

**Epoch Complete!**

When all batches in the dataset have been processed, one **epoch** is completed.

- All samples in the dataset have been **seen once by the model**
- Model **weights are updated multiple times** (once for each batch)
- DataLoader **workers remain active** and are reused for the next epoch
- If `shuffle=True`, the dataset is **reshuffled to create a new order**

↓ **Next epoch starts**

**Creating DataLoaders**

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=32,
    num_workers=0,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=32,
    num_workers=0,
    shuffle=False
)

In short: DataLoader(dataset, batch_size, shuffle) → Iterable batches of tensors [N, 3, 64, 64]

Iterating Through DataLoaders

In [ ]:
# Get a single batch
img_batch, label_batch = next(iter(train_dataloader))

print(f"Image batch shape: {img_batch.shape}")
# Output: [32, 3, 64, 64] -> [batch, channels, height, width]

print(f"Label batch shape: {label_batch.shape}")
# Output: [32] -> one label per image

**Why batches are efficient:**

1. GPU Parallelism - GPUs excel at processing multiple images simultaneously
2. Stable Gradients - Averaging gradients across a batch reduces noise
3. Memory Efficiency - Process multiple samples in one forward/backward pass
4. Faster Training - Batch processing is much faster than one-at-a-time





---

## 1️⃣ GPU Parallelism

**PyTorch বা deep learning এ GPU ব্যবহার করা হয়।**

GPU একসাথে **অনেক data process করতে পারে**।

উদাহরণ:

* 1 image → GPU পুরো ক্ষমতা ব্যবহার হয় না
* 32 images batch → GPU **একসাথে 32টা image process করে**

তাই training **দ্রুত হয়**।

---

## 2️⃣ Stable Gradients

Training এর সময় model **gradient** ব্যবহার করে weights update করে।

যদি **একটা image** দিয়ে update করা হয়:

* gradient **noisy** হতে পারে
* learning unstable হতে পারে

কিন্তু **batch (যেমন 32 images)** ব্যবহার করলে:

* সব gradient **average হয়**
* training **stable হয়**

---

## 3️⃣ Memory Efficiency

Batch ব্যবহার করলে:

* অনেক sample **এক forward pass এ process হয়**
* তারপর **এক backward pass এ gradient calculate হয়**

মানে:

```text
32 images → 1 forward pass → 1 backward pass
```

এতে **memory efficient হয়**।

---

## 4️⃣ Faster Training

যদি একবারে **একটা image** দিয়ে training করো:

```text
image1 → update
image2 → update
image3 → update
```

কিন্তু batch হলে:

```text
32 images → update
```

তাই **batch processing অনেক দ্রুত**।

---



## Part 5: Tensor Shape Convention

### PyTorch Image Tensor Format

In **PyTorch**, image tensors follow the **NCHW** format:

| Dimension | Meaning      | Example       |
|-----------|--------------|---------------|
| N         | Batch size   | 32            |
| C         | Color channels | 3 (RGB)     |
| H         | Height       | 64            |
| W         | Width        | 64            |

---


In [ ]:
# Single image tensor: [C, H, W]
img.shape  # [3, 64, 64]

# Batch of images: [N, C, H, W]
batch.shape  # [32, 3, 64, 64]

Matplotlib Requires Different Format

To plot tensors with matplotlib, we need to permute dimensions:

In [ ]:
# PyTorch format: [C, H, W] -> [3, 64, 64]
# Matplotlib format: [H, W, C] -> [64, 64, 3]

img_permuted = img.permute(1, 2, 0)  # Reorder dimensions
plt.imshow(img_permuted)

## Key Takeaways: Concept Summary

| Concept                  | Purpose                          | Key Points |
|---------------------------|---------------------------------|------------|
| `transforms.Compose()`    | Chain multiple transforms together | Execute transforms in sequence |
| `transforms.Resize()`     | Make all images same size        | Required for batch processing |
| `transforms.ToTensor()`   | Convert to tensor, scale to [0, 1] | Changes shape to [C, H, W] |
| `ImageFolder`             | Auto-load images and labels      | Expects folder structure |
| `DataLoader`              | Create batches, shuffle, parallel loading | Makes data iterable |
| NCHW format               | `[Batch, Channels, Height, Width]` | PyTorch convention |


What We Learned

- Transform Pipeline - Resize → Augment → ToTensor workflow
- ImageFolder - Automatic image loading from directory structure
- DataLoader - Efficient batching and parallel loading
- Tensor Shapes - Understanding [C, H, W] vs [N, C, H, W] formats
- Data Augmentation - Only apply to training data, not test data
- Performance - Use num_workers for faster data loading

Setting Up the Environment

Now we will set up the environment for this lab, then run the notebook to practice each step.

Step 1: Install Jupyter Extension

Open VS Code Server provided by

Poridhi

, go to Extensions (Ctrl+Shift+X), and install the

Jupyter

extension.

![image](https://raw.githubusercontent.com/poridhiEng/lab-asset/c86bc88e676d50722669abf52c9c25213adc5b70/tensorcode/Deep-learning-with-pytorch/Classification/Lab_01/images/image-2.png)

Step 2: Create Virtual Environment

In [ ]:
sudo apt install python3.12-venv
python3 -m venv venv
source venv/bin/activate

Step 3: Download the Notebook

In [1]:
!curl -O https://raw.githubusercontent.com/poridhioss/pytorch-labs-resources/refs/heads/main/Custom-Datasets-with-pytorch/Lab_02/lab_02.ipynb

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    14  100    14    0     0     30      0 --:--:-- --:--:-- --:--:--    30


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

It will download the notebook to your current directory from

Poridhi's GitHub repository

.

Step 4: Open and Run the Notebook

Open

lab_02.ipynb

in VS Code. In the top right corner of the notebook, you will see a button

Select Kernel

. Click on it, then a popup will open to select

Python Environment

. After selecting the Python environment for the first time, it will ask permission to install

python ipykernel

. Click on

Install

button to install the

python ipykernel

. Once installed, you will see the

venv

virtual environment in the list of kernels. Select it as the kernel.

![image](https://github.com/poridhiEng/lab-asset/blob/main/tensorcode/Deep-learning-with-pytorch/Classification/Lab_01/images/image-3.png?raw=true)

Next Steps

In

Lab 3

, we'll learn how to build our own custom

Dataset

class from scratch!